# Logistics ML Model with API — Training & Evaluation Notebook

**Purpose:** Document and reproduce the end-to-end machine-learning workflow for predicting late deliveries in a fictional cold-chain logistics scenario.

> **Dataset disclosure:** This project uses synthetically generated shipment and warehouse-scan data. It does not contain real company records. Business-cost assumptions are illustrative.

## Workflow

1. Generate synthetic data.
2. Clean shipment and warehouse-scan records.
3. Explore the data.
4. Engineer dispatch-time features and create a chronological train/test split.
5. Train and compare four classifiers.
6. Evaluate the selected model.
7. Optimize the decision threshold using training-only out-of-fold predictions.
8. Explain predictions with SHAP.
9. Monitor input-feature drift.
10. Serve predictions through FastAPI and Streamlit (deployment is covered in the project README).

The notebook uses the existing `src/` scripts so that the documented workflow stays aligned with the tested implementation.


## 1. Set up the project paths

Run this notebook from the project root or from the `notebooks/` directory. The helper below locates the project root by checking for the `src/` and `data/` folders.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import json

PROJECT_ROOT = Path.cwd()
if not ((PROJECT_ROOT / "src").exists() and (PROJECT_ROOT / "data").exists()):
    PROJECT_ROOT = PROJECT_ROOT.parent

if not ((PROJECT_ROOT / "src").exists() and (PROJECT_ROOT / "data").exists()):
    raise FileNotFoundError(
        "Could not locate the project root. Open this notebook from the Logistics-ML-Model folder."
    )

os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)
print("Python:", sys.version)


## 2. Install / verify dependencies

The project dependencies are listed in `requirements.txt`. In the project's activated virtual environment, install them with:

```bash
python -m pip install -r requirements.txt
```

The cell below checks the core libraries without reinstalling packages.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)


## 3. Run the data-generation and preprocessing workflow

The following scripts are the source of truth for the project's data workflow. Execute them in order when you want to regenerate the synthetic dataset and processed splits.

**Note:** Running these cells regenerates the synthetic data and may overwrite existing CSV files and reports. If you only want to inspect the existing trained model, skip to Section 5.


In [ ]:
RUN_DATA_PIPELINE = False  # Change to True to regenerate and preprocess the synthetic data.

data_scripts = [
    "generate_data.py",
    "clean_data.py",
    "eda.py",
    "prepare_data.py",
]

if RUN_DATA_PIPELINE:
    for script in data_scripts:
        print(f"\\n===== Running {script} =====")
        subprocess.run(
            [sys.executable, str(PROJECT_ROOT / "src" / script)],
            cwd=PROJECT_ROOT,
            check=True,
        )
else:
    print("Skipped data regeneration. Using existing files under data/processed/.")


## 4. Inspect the processed training and test data

The split is chronological rather than random to reduce temporal leakage. The target is `late_delivery`; shipment identifiers and post-delivery information are not model inputs.


In [ ]:
train_path = PROJECT_ROOT / "data" / "processed" / "train.csv"
test_path = PROJECT_ROOT / "data" / "processed" / "test.csv"

if not train_path.exists() or not test_path.exists():
    raise FileNotFoundError(
        "Processed train.csv/test.csv not found. Run the data pipeline in Section 3."
    )

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)
display(train_df.head())


In [ ]:
TARGET = "late_delivery"

print("Training target distribution:")
display(train_df[TARGET].value_counts().rename("count").to_frame())

print("\\nTest target distribution:")
display(test_df[TARGET].value_counts().rename("count").to_frame())

print("\\nMissing values in training data:")
display(train_df.isna().sum().sort_values(ascending=False).head(10).to_frame("missing_count"))


## 5. Train and compare the machine-learning models

The training script builds preprocessing pipelines and evaluates Logistic Regression, Decision Tree, Random Forest and XGBoost using time-series cross-validation and randomized hyperparameter search.

The best fitted pipeline is saved to `models/best_model.joblib`.

**This step can take several minutes.** Set `RUN_MODEL_TRAINING = True` only when you want to retrain the models.


In [ ]:
RUN_MODEL_TRAINING = False  # Change to True to run RandomizedSearchCV again.

if RUN_MODEL_TRAINING:
    subprocess.run(
        [sys.executable, str(PROJECT_ROOT / "src" / "train_models.py")],
        cwd=PROJECT_ROOT,
        check=True,
    )
else:
    print("Skipped model training. Loading the existing model and comparison report.")


In [ ]:
comparison_path = PROJECT_ROOT / "reports" / "model_comparison.csv"
if comparison_path.exists():
    model_comparison = pd.read_csv(comparison_path)
    display(model_comparison)
else:
    print("Model comparison report not found. Run Section 5 with RUN_MODEL_TRAINING=True.")


## 6. Evaluate the selected model

The evaluation script reports test-set classification metrics at the default threshold and at the business-optimized threshold. The test set is used for final evaluation, not for selecting the threshold.


In [ ]:
RUN_FINAL_EVALUATION = False  # Change to True to regenerate evaluation reports.

if RUN_FINAL_EVALUATION:
    subprocess.run(
        [sys.executable, str(PROJECT_ROOT / "src" / "evaluate_model.py")],
        cwd=PROJECT_ROOT,
        check=True,
    )

evaluation_path = PROJECT_ROOT / "reports" / "evaluation_results.json"
if evaluation_path.exists():
    evaluation_results = json.loads(evaluation_path.read_text(encoding="utf-8"))
    print(json.dumps(evaluation_results, indent=2))
else:
    print("Evaluation report not found. Run src/evaluate_model.py.")


## 7. Business-cost-based threshold optimization

The threshold-optimization script uses time-series out-of-fold predictions from the training data and illustrative costs (false negative: ₹1,000; false positive: ₹200). This avoids choosing the threshold using the held-out test labels.

The selected threshold in the current project run is **0.33**. The lower threshold increases late-shipment recall while also increasing false positives.


In [ ]:
threshold_path = PROJECT_ROOT / "reports" / "optimal_threshold.json"
if threshold_path.exists():
    threshold_results = json.loads(threshold_path.read_text(encoding="utf-8"))
    print(json.dumps(threshold_results, indent=2))
else:
    print("Threshold report not found. Run src/optimize_threshold.py.")


## 8. Explainability with SHAP

The SHAP script explains global feature importance and an individual shipment prediction. It produces summary, feature-importance and waterfall plots under `reports/figures/`.

SHAP values describe how model features contribute to the model output; they do not prove causation.


In [ ]:
RUN_SHAP_EXPLANATION = False  # Change to True to regenerate SHAP explanations.

if RUN_SHAP_EXPLANATION:
    subprocess.run(
        [sys.executable, str(PROJECT_ROOT / "src" / "explain_model.py")],
        cwd=PROJECT_ROOT,
        check=True,
    )

shap_importance_path = PROJECT_ROOT / "reports" / "shap_feature_importance.csv"
if shap_importance_path.exists():
    shap_importance = pd.read_csv(shap_importance_path)
    display(shap_importance.head(10))
else:
    print("SHAP report not found. Run src/explain_model.py.")


## 9. Monitor feature drift

The drift script compares the training feature distribution with the chronological test feature distribution using the Population Stability Index (PSI).

- PSI < 0.10: no significant drift
- PSI from 0.10 to 0.20: moderate drift
- PSI ≥ 0.20: significant drift

A shift in `dispatch_month` is expected because the training and test periods cover different portions of the year. A PSI alert indicates a distribution change; it does not by itself prove model performance has degraded.


In [ ]:
RUN_DRIFT_MONITORING = False  # Change to True to regenerate the drift report.

if RUN_DRIFT_MONITORING:
    subprocess.run(
        [sys.executable, str(PROJECT_ROOT / "src" / "monitor_drift.py")],
        cwd=PROJECT_ROOT,
        check=True,
    )

drift_path = PROJECT_ROOT / "reports" / "drift_report.json"
if drift_path.exists():
    drift_report = json.loads(drift_path.read_text(encoding="utf-8"))
    print(json.dumps(drift_report, indent=2))
else:
    print("Drift report not found. Run src/monitor_drift.py.")


## 10. Review saved visualizations

The EDA and explainability scripts save figures under `reports/figures/`. The following cell lists the available figures.


In [ ]:
figures_dir = PROJECT_ROOT / "reports" / "figures"
if figures_dir.exists():
    figure_files = sorted(figures_dir.glob("*.png"))
    print(f"Found {len(figure_files)} figure(s):")
    for file in figure_files:
        print("-", file.relative_to(PROJECT_ROOT))
else:
    print("No reports/figures directory found.")


## 11. Key results from the current project run

The recorded evaluation from the completed project run:

| Metric | Default threshold (0.50) |
|---|---:|
| Accuracy | 72.03% |
| Precision | 48.21% |
| Recall | 72.61% |
| F1 score | 57.95% |
| ROC-AUC | 79.87% |
| PR-AUC | 62.06% |

The dummy baseline PR-AUC was 26.54%, so the model's PR-AUC was approximately 133.82% higher on a relative basis.

At the optimized threshold of 0.33, test recall was 85.49% and estimated test cost was ₹10,25,600, compared with ₹11,41,000 at threshold 0.50. These cost figures use illustrative assumptions and are not actual business savings.

## 12. Deployment

The application is served by two Docker Compose services:

- **FastAPI:** `http://localhost:8000` and Swagger UI at `http://localhost:8000/docs`
- **Streamlit:** `http://localhost:8501`

From the project root:

```bash
docker compose up -d
docker compose ps
```

To stop the services:

```bash
docker compose down
```

## 13. Limitations and responsible interpretation

- The dataset is synthetic and is suitable for demonstration, not operational decision-making.
- Model probabilities are estimates, not guarantees.
- The cost matrix is illustrative and should be replaced with validated business costs before real deployment.
- Input drift monitoring is not a substitute for monitoring prediction quality against confirmed outcomes.
- Real deployment requires validation on representative, authorized operational data.
